<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 4 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">湖表与内部表关联</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">候选实验：需要讲师预置真实 Iceberg 环境，当前尚未实测。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">目标 Doris 4.1.3 · 订单数据 · 独立实验库</span>
</div>

具备讲师预置的 Iceberg 环境后，你将直查十笔湖上订单、关联内部客户表，并对导入结果核对十行、金额 12220.60。当前候选实验尚未实测，没有外部环境时请先阅读。

[讲义](course4_querying_external_data.md) · [课程入口](../README.md)


## 环境要求与状态

这是需要外部环境的候选 Lab，本实验不提供 Iceberg 服务部署。讲师需预置包含 datasets/wwi/sample.json 中 orders 的六个字段与十行数据的 Iceberg orders 表，并通过 Doris Catalog 可查询。设置 DW_ICEBERG_ORDERS=catalog.database.table；没有该环境时应明确记录未执行，不用内部表冒充湖表。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

import os
from dw_course.runtime import identifier
source_parts = os.environ["DW_ICEBERG_ORDERS"].split(".")
if len(source_parts) != 3:
    raise ValueError("Expected catalog.database.table")
source = ".".join(identifier(part) for part in source_parts)
from dw_course import WarehouseLab
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.wwi import HISTORY_COLUMNS as ORDER_COLUMNS, history_ddl as order_ddl, history_rows, sample
from dw_course.ui import show_sql, show_response

lab = WarehouseLab()




## 1. 直查湖表

同一份 10 笔订单应有 12220.60 金额。这里没有导入数据；Catalog 提供外部表元数据。


In [ ]:
expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {source}"), [(10,"12220.60")])
lab.sql(f"EXPLAIN SELECT * FROM {source} WHERE order_id = 1");


## 2. 与内部客户表关联

每个订单客户对应一个客户行；关联后数量不能扩大。仅重建 customers_sample 与 orders_from_lake。


In [ ]:
lab.execute("DROP TABLE IF EXISTS customers_sample")
lab.execute('CREATE TABLE customers_sample (customer_id BIGINT, customer_name STRING) UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.insert("customers_sample", ["customer_id", "customer_name"],
           [(r["customer_id"], r["customer_name"]) for r in sample()["customers"]])
expect(lab.query(f"SELECT COUNT(*), SUM(o.order_amount) FROM {source} o JOIN customers_sample c ON o.customer_id=c.customer_id"),
       [(10, "12220.60")])
lab.execute("DROP TABLE IF EXISTS orders_from_lake")
ddl = order_ddl("orders_from_lake")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.execute(f"INSERT INTO orders_from_lake ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM {source}")
expect(lab.query("SELECT order_id, customer_id, CAST(order_date AS STRING), order_amount, line_count, data_source FROM orders_from_lake ORDER BY order_id"),
       history_rows())
lab.close()


## 完成与边界

记录 Catalog 类型和外部服务版本、原表标识、结果与计划。本实验不执行独立 Parquet 的 S3 TVF 查询；读取成功不代表外部写入、Schema 演进或性能 SLA 已验证。


## 自己动手

说明为什么把 Parquet 放进对象存储还不等于创建 Iceberg 表。湖表使用 WWI 原日期，内部客户维表按 customer_id 唯一，关联不能扩大订单行数。没有真实外部 Catalog 时，本 Lab 仍为未执行。
